[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/01_eda_visualization/01_eda_visualization.ipynb)

# 01. 데이터 탐색과 시각화 — 모델링 전에 데이터를 읽는 법

`ml-curriculum`에서 다룬 Iris나 MNIST 같은 교과서 데이터셋은 **이미 깨끗하게 정리된 상태**로 제공됩니다.
[결측치](https://github.com/karzit/temp/blob/master/glossary.md#missing-value)가 없고, 모든 값이 숫자이고, 말이 안 되는 값도 없습니다. 그래서 `load_iris()` 다음 줄에서 바로
`fit()`을 호출할 수 있었습니다.

현실의 표 데이터는 그렇지 않습니다.

- 값이 비어 있는 칸이 있습니다 (**결측치**)
- 택시가 시속 4,834마일로 달렸다고 기록되어 있습니다 (**[이상치](https://github.com/karzit/temp/blob/master/glossary.md#outlier)**)
- `"Manhattan"`, `"credit card"` 같은 **문자열**이 들어 있어 모델에 그대로 넣을 수 없습니다
- 같은 정보를 담은 컬럼이 두 개씩 있습니다 (**중복**)

이런 데이터를 다루는 첫 단계가 **EDA([탐색적 데이터 분석](https://github.com/karzit/temp/blob/master/glossary.md#eda), Exploratory Data Analysis)** 입니다.
목표는 예쁜 그래프를 그리는 것이 아니라 **"이 데이터를 모델에 넣으려면 무엇을 고쳐야 하는가"의
목록을 만드는 것**입니다. 이 노트북 마지막에 그 목록을 정리하고, 다음 노트북에서 하나씩 처리합니다.

## 이 노트북의 구성

**1부에서는 택시 데이터로 EDA를 끝까지 한 바퀴 돕니다.** 데이터를 불러와 첫인상을 살피고,
그래프로 분포와 관계를 확인하고, 전처리 계획표를 만드는 데까지 갑니다.

**2부에서는 타이타닉을 봅니다.** 결측치가 77%인 컬럼, 생존 여부처럼 결과와 직결된 범주 —
택시 데이터에는 없는 상황들을 다루기 위해서입니다.

| | 데이터 | 다루는 것 |
|---|---|---|
| **1부** | `trips` — 뉴욕 택시 운행 기록 6,433건 (2019년 3월) | EDA의 기본 흐름 전체: 첫인상 → 그래프 → [상관계수](https://github.com/karzit/temp/blob/master/glossary.md#correlation) → 전처리 계획 |
| **2부** | `titanic` — 타이타닉 탑승자 891명 | 결측치가 77%일 때의 판단, 범주별 비교(`boxplot`·`hue`) |

두 데이터는 예측 대상과 문제 유형이 다릅니다. 시리즈 전체(02~04번)에서 **회귀 예제는 택시,
분류 예제는 타이타닉**으로 계속 쓰입니다.

| 이름 | 예측 대상 | 문제 유형 |
|---|---|---|
| `trips` | `duration` — 이동 시간(분) | **회귀** (숫자 맞히기) |
| `titanic` | `survived` — 생존 여부(0/1) | **분류** (범주 맞히기) |

둘 다 seaborn에 내장된 공개 데이터셋이고, 결측치·이상치·문자열 범주를 실제로 가지고 있습니다.

## 이 노트북에서 배우는 것

**1부 — 택시**

1. `head`/`info`/`describe`로 데이터의 첫인상 잡기 — **`info()`와 `describe()`가 각각 무엇을 알려주는지**
2. 상황에 맞는 그래프 고르기 — 변수가 몇 개인지, 범주형인지 수치형인지로 결정됩니다
3. `countplot`, `histplot`, `scatterplot`, `jointplot`, `heatmap`
4. `plt.subplots`로 그래프를 나란히 놓고 비교하기
5. 관찰한 내용을 **전처리 계획**으로 옮기기

**2부 — 타이타닉**

6. 결측치가 20~80%일 때 **채울지 버릴지 판단하기**
7. `boxplot`과 `hue`로 **범주에 따라 수치가 어떻게 달라지는지** 보기
8. 같은 [데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage) 질문을 **다른 데이터에 다시 적용**해보기

**소요 시간**: 1부(택시) 40~50분, 2부(타이타닉) 25~35분. 모든 셀이 몇 초 안에 끝나므로
**1부까지만 하고 끊었다가 이어서 봐도 됩니다.**

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`). 아래쪽 셀은 위쪽 셀에서 만든
  변수를 그대로 쓰기 때문에, 중간부터 실행하면 `NameError`가 납니다.
- **실행 결과는 저장되어 있지 않습니다.** 코드 셀 아래가 비어 있는 것이 정상이고,
  직접 실행해야 표와 그래프가 나타납니다.
- 본문에 적힌 숫자(예: "MAE 8.33분")는 **실행하면 나오는 값**입니다. 글을 읽으면서
  그 숫자가 어느 셀의 출력인지 짚어보면 이해가 빠릅니다.
- 코드는 **그대로 실행만 해도 되지만**, 숫자를 바꿔 다시 실행해보는 것이 가장 좋은 연습입니다.
- **pandas 문법이 막히면** [00_pandas_for_tabular](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에
  이 시리즈에서 쓰는 문법만 모아뒀습니다(`pd.to_datetime`, `get_dummies`, `dropna` …). 사전처럼 찾아보세요.
- 코드 셀 뒤의 **결과 읽는 법**은 그 셀의 출력을 어떻게 읽는지 알려줍니다.
  숫자가 예상과 다르면 거기부터 보세요.
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 데이터 로드 셀에서 `URLError` / `HTTPError` | `sns.load_dataset()`은 데이터를 **인터넷에서 받아옵니다** | 네트워크 연결 확인. 사내망이라면 프록시 때문일 수 있습니다 |
| 그래프가 안 뜨고 `<Axes: ...>` 텍스트만 나온다 | 셀 마지막 줄이 그래프 객체를 반환 | `plt.show()`를 마지막 줄에 추가 |
| 본문에 적힌 숫자(6,433건 등)와 내 출력이 다르다 | seaborn 버전에 따라 예제 데이터가 조금 다를 수 있음 | **숫자보다 "어떻게 읽는가"에 집중하세요.** 결론은 바뀌지 않습니다 |
| `pairplot`/`heatmap`이 오래 걸린다 | 컬럼이 많으면 조합이 급격히 늘어남 | `vars=[...]`로 볼 컬럼을 3~4개로 줄이기 |
| 2부(타이타닉)에서 `NameError` | 1부 준비 셀을 실행하지 않음 | 위에서부터 순서대로 실행 |

여기 없는 문제(설치 실패, 한글 깨짐, GPU 설정)는 [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)에 모아뒀습니다.

## Colab에서 열기

맨 위 배지를 클릭하면 바로 열립니다. 아래 첫 셀이 Colab인지 로컬인지 감지해 필요한 패키지를 설치합니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib

### 준비 셀 — 라이브러리와 한글 폰트

아래 셀은 네 노트북에 공통으로 들어가는 준비 코드입니다. **내용을 이해할 필요는 없고 그냥
실행**하면 됩니다. `numpy`·`pandas`·`matplotlib`·`seaborn`을 불러오고, 그래프의 한글이
깨지지 않게 폰트를 잡고, 결과가 매번 같도록 무작위 시드(`RANDOM_STATE = 42`)를 고정합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")   # seaborn 기본 스타일 지정 — 흰 배경에 옅은 격자

RANDOM_STATE = 42

---

# 1부. 택시 데이터로 EDA 한 바퀴

**여기서는 `trips`(뉴욕 택시) 하나만 봅니다.** 데이터를 불러와서 전처리 계획표를 만들 때까지,
EDA의 기본 흐름을 처음부터 끝까지 한 번 통과합니다.

| 절 | 배우는 것 |
|---|---|
| 1~2 | `head`/`info`/`describe`로 결측치·이상치·자료형 파악 |
| 3~4 | 그래프 고르는 기준, `countplot`·`histplot` |
| 5~6 | `scatterplot`/`jointplot`, 상관계수 `heatmap`, 데이터 누출 |
| 7~8 | `subplots`로 나란히 비교하기, 전처리 계획표 만들기 |

타이타닉은 2부에서 불러옵니다.

---

## 1. 데이터 불러오기

`trips`에서는 원본에 없던 컬럼 4개를 직접 계산해서 추가합니다. **원본 컬럼을 조합해 새 컬럼을 만드는
것을 [파생 변수](https://github.com/karzit/temp/blob/master/glossary.md#feature-engineering)(feature engineering)** 라고 하고, 모델 성능에 가장 크게 기여하는 작업 중 하나입니다.

- 승차·하차 **시각**은 그대로는 쓸 수 없지만, 둘을 빼면 **이동 시간**이 나옵니다 → 이것이 예측 대상
- 승차 시각에서 **요일**과 **시간대**를 뽑아내면 "출퇴근 시간엔 오래 걸린다" 같은 패턴을 모델이 학습할 수 있습니다

> `pd.to_datetime`, `.dt.total_seconds()`, `.dt.dayofweek`가 정확히 무슨 일을 하는지는
> [00번 3절](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에서 하나씩 실행해볼 수 있습니다.

In [ ]:
# [회귀용] 뉴욕 택시 운행 기록 — 이동 시간(분)을 맞히는 문제로 씁니다.
trips = sns.load_dataset("taxis")   # load_dataset: seaborn이 제공하는 예제 데이터를 인터넷에서 받아 DataFrame으로 돌려준다

# 승·하차 시각은 문자열로 들어옵니다. datetime으로 바꿔야 뺄셈을 할 수 있습니다.
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])

# 원본에 없는 정보를 계산해서 새 컬럼으로 만드는 것을 "파생 변수(feature engineering)"라고 합니다.
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60  # 이동 시간(분) — 예측 대상
trips["speed"] = trips["distance"] / (trips["duration"] / 60)                     # 평균 시속(mph)
trips["weekday"] = trips["pickup"].dt.dayofweek                                   # 요일 (0=월 … 6=일)
trips["hour"] = trips["pickup"].dt.hour                                           # 승차 시각(0~23시)

print("trips:", trips.shape)

### `trips` 컬럼 설명

| 컬럼 | 설명 |
|---|---|
| `pickup` / `dropoff` | 승차 시각 / 하차 시각 |
| `passengers` | 승객 수 |
| `distance` | 이동 거리 (mile) |
| `fare` / `tip` / `tolls` / `total` | 기본요금 / 팁 / 통행료 / 총액 |
| `color` | 택시 종류 (yellow / green) |
| `payment` | 결제 수단 (credit card / cash) |
| `pickup_zone` / `dropoff_zone` | 승차 / 하차 세부 지역 (200개 이상) |
| `pickup_borough` / `dropoff_borough` | 승차 / 하차 자치구 (승차 4개 · 하차 5개) |
| `duration` | *(파생)* 이동 시간(분) — **예측 대상** |
| `speed` | *(파생)* 평균 시속 (mph) |
| `weekday` / `hour` | *(파생)* 승차 요일(0=월) / 시각 |

## 2. 첫인상 — `head` / `info` / `describe`

데이터를 받으면 항상 이 세 가지를 먼저 봅니다. 각각 알려주는 것이 다릅니다.

### `head()` — 값이 실제로 어떻게 생겼는가

컬럼 이름만 보고 짐작한 것과 실제 값이 다른 경우가 많습니다. `head()`는 기본 5행을 보여주고,
`head(3)`처럼 개수를 지정할 수 있습니다.

In [ ]:
trips.head()

### `info()` — 결측치와 자료형

**전처리에서 무엇을 해야 할지는 사실상 `info()` 하나로 결정됩니다.** 두 가지를 봅니다.

1. **Non-Null Count가 전체 행 수보다 작은 컬럼** → 결측치가 있다 → 채우거나 지워야 한다
2. **Dtype이 `object`인 컬럼** → 문자열이다 → 숫자로 바꿔야 모델에 넣을 수 있다

`Dtype`의 의미는 이렇습니다.

| Dtype | 뜻 | 모델에 바로 넣을 수 있나 |
|---|---|---|
| `int64` / `float64` | 정수 / 실수 | ✅ |
| `bool` | 참/거짓 | ✅ (내부적으로 1/0) |
| `object` | 문자열 등 | ❌ **인코딩 필요** |
| `datetime64` | 날짜·시각 | ❌ 연·월·요일 등으로 쪼개서 사용 |
| `category` | 범주형 | ❌ 인코딩 필요 |

In [ ]:
trips.info()

**결과 읽는 법**

`trips`를 보면 `payment`, `pickup_zone`, `dropoff_zone`, `pickup_borough`, `dropoff_borough`의
Non-Null Count가 6,433보다 작습니다. 결측치가 있다는 뜻입니다. 파생 변수인 `speed`도 6건 비어 있는데,
`duration`이 0이라 0을 0으로 나눈 결과입니다. 개수만 따로 세려면 이렇게 합니다.

`isnull()`은 각 칸이 비었는지를 True/False로 바꾸고, `sum()`은 True를 1로 세어 컬럼별 개수를 냅니다.
(결측치 관련 문법은 [00번 4절](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/00_pandas_for_tabular/00_pandas_for_tabular.ipynb) 참고)

In [ ]:
trips.isnull().sum()

**결과 읽는 법** — 택시의 결측치는 **전부 합쳐도 1%대**입니다(가장 많은 컬럼이 6,433건 중 45건). 이 정도면
고민할 것 없이 **해당 행을 지우면 끝**이고, 02번에서 그렇게 처리합니다.

> 판단이 필요한 것은 결측이 20%, 80%씩 되는 경우입니다. 지우면 데이터가 남아나지 않고, 채우면
> 대부분이 추측값이 되기 때문입니다. **2부의 타이타닉에서 그 상황을 다룹니다.**

### `describe()` — 말이 안 되는 값 찾기

수치형 컬럼의 개수·평균·표준편차·최소·사분위수·최대를 보여줍니다.
**여기서 가장 중요한 건 min과 max입니다.** 상식적으로 불가능한 값이 있으면 이상치입니다.

In [ ]:
trips.describe()

**결과 읽는 법** — 눈에 띄는 것 세 가지입니다.

| 관찰 | 문제 |
|---|---|
| `speed`의 max가 **4834** | 택시가 시속 4,834마일로 달릴 수는 없습니다 |
| `duration`의 min이 **0** | 이동 시간이 0분 |
| `passengers`의 min이 **0** | 승객이 0명인 운행 |

`speed`가 왜 저렇게 나오는지 실제 행을 열어보면 원인이 분명해집니다.

In [ ]:
# nlargest(n, 컬럼): 그 컬럼 기준 상위 n개 행. sort_values().head()와 같지만 더 짧다
trips.nlargest(3, "speed")[["distance", "duration", "speed"]]

**결과 읽는 법** — 9.4마일을 **0.12분(7초)** 만에 갔다고 기록되어 있습니다. 물리적으로 불가능하니 미터기 오작동이나
기록 오류입니다. `speed`는 `distance / duration`으로 계산한 값이라, **분모인 `duration`이 0에
가까워지면 값이 폭발**합니다.

이런 행은 모델 학습을 크게 망칩니다. 특히 뒤에서 다룰 신경망은 손실을 제곱해서 계산하기 때문에,
극단값 몇 개가 전체 학습 방향을 끌고 가버립니다. **이상치 처리가 필요한 이유**입니다.

`describe()`는 기본적으로 수치형만 보여줍니다. **문자열 컬럼의 요약**을 보려면 `include="object"`를 줍니다.
`unique`(고유값 개수)와 `top`(최빈값)이 나오는데, **`unique`가 특히 중요합니다.** 뒤에서 문자열을
0/1 컬럼으로 펼칠 때 고유값 개수만큼 컬럼이 늘어나기 때문입니다.

In [ ]:
trips.describe(include="object")

**결과 읽는 법**

`pickup_zone`은 고유값이 194개, `dropoff_zone`은 203개입니다. 이걸 그대로 0/1 컬럼으로 펼치면
컬럼이 400개 가까이 늘어납니다. 반면 `pickup_borough`는 4개(하차는 5개)뿐이라 부담이 없습니다.
**같은 정보를 더 거친 단위로 담은 컬럼이 있으면 그쪽을 쓰는 편이 낫습니다.**

---

## 3. 어떤 그래프를 그릴 것인가

그래프 종류는 **"변수를 몇 개 보는가"** 와 **"그 변수가 범주형인가 수치형인가"** 로 정해집니다.
이 표만 익혀두면 상황마다 헤매지 않습니다.

| 보려는 것 | 변수 유형 | 함수 |
|---|---|---|
| 한 변수의 분포 | 범주형 | `countplot` |
| 한 변수의 분포 | 수치형 | `histplot` |
| 두 변수의 관계 | 범주형 + 수치형 | `boxplot`, `violinplot`, `barplot` **(2부)** |
| 두 변수의 관계 | 수치형 + 수치형 | `scatterplot`, `jointplot` |
| 그룹별로 나눠 보기 | 위 그래프에 `hue=` 추가 **(2부)** | — |
| 여러 수치형의 관계 한 번에 | 수치형 여러 개 | `heatmap`(상관계수), `pairplot` |

**범주형(categorical)** 은 값의 종류가 정해져 있는 변수입니다 (`Manhattan`/`Queens`, `yellow`/`green`).
**수치형(numeric)** 은 크기 비교와 산술 연산이 의미 있는 변수입니다 (`distance`, `fare`).

**(2부)** 표시가 붙은 것은 타이타닉에서 다룹니다. 범주에 따라 값이 얼마나 달라지는지 보려면
생존 여부·객실 등급처럼 결과와 직결된 범주가 있어야 하는데, 택시의 범주형은 자치구·결제 수단
정도라 차이가 잘 드러나지 않습니다.

> 숫자로 저장되어 있어도 범주형인 경우가 있습니다. `weekday`(0~6)는 숫자지만 "월요일 + 화요일 = 수요일"이
> 말이 안 되므로 범주형입니다. 2부에 나올 `pclass`(1/2/3)도 마찬가지입니다.

## 4. 한 변수 보기

### `countplot` — 범주형의 개수 세기

각 범주가 몇 번 나타나는지 막대로 보여줍니다. `x=`에 넣으면 세로 막대, `y=`에 넣으면 가로 막대인데,
**범주 이름이 길면 `y=`가 훨씬 읽기 편합니다.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))   # subplots(행, 열): 그래프 칸을 격자로 만든다. figsize는 (가로, 세로) 인치

sns.countplot(data=trips, x="pickup_borough", ax=axes[0])
axes[0].set_title("승차 자치구")

sns.countplot(data=trips, x="payment", ax=axes[1])
axes[1].set_title("결제 수단")

plt.tight_layout()   # 라벨이 서로 겹치지 않도록 칸 간격을 자동 조정
plt.show()

**결과 읽는 법** — 승차의 대부분이 맨해튼에 몰려 있습니다. 정확한 숫자는 `value_counts()`로 확인합니다.
**`value_counts()`는 기본적으로 결측치를 세지 않으므로**, 결측치까지 보려면 `dropna=False`를 줍니다.

In [ ]:
# value_counts: 각 값이 몇 번 나오는지 센다. dropna=False면 결측치도 하나의 값으로 센다
print(trips["pickup_borough"].value_counts(dropna=False))
print()
print("비율(%):")
print((trips["pickup_borough"].value_counts(normalize=True) * 100).round(1))

**결과 읽는 법** — 맨해튼이 82%로 압도적입니다. 이렇게 **한 범주에 심하게 쏠린 컬럼**은 모델 입장에서 정보량이 적습니다
(거의 모든 행에서 같은 값이므로 구분에 도움이 안 됨). 그렇다고 버릴 필요는 없고, 나중에
[변수중요도](https://github.com/karzit/temp/blob/master/glossary.md#feature-importance)를 볼 때 실제로 기여가 낮게 나오는지 확인하면 됩니다.

### `histplot` — 수치형의 분포

수치형은 값의 종류가 너무 많아서 `countplot`을 쓸 수 없습니다. 대신 값의 범위를 **구간(bin)** 으로
나눠 각 구간에 몇 개가 들어가는지 셉니다.

`bins`로 구간 개수를 조절합니다. 너무 적으면 뭉뚱그려지고, 너무 많으면 들쭉날쭉해집니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, b in zip(axes, [10, 30, 100]):
    sns.histplot(data=trips, x="duration", bins=b, ax=ax)
    ax.set_title(f"bins={b}")

plt.tight_layout()
plt.show()

**결과 읽는 법** — 세 그래프 모두 **오른쪽으로 길게 늘어진(right-skewed)** 모양입니다. 짧은 운행이 대부분이고
아주 긴 운행이 드물게 있다는 뜻입니다. 요금·소득·대기시간처럼 "0 아래로는 못 가고 위로는 열려 있는"
값에서 흔히 나타나는 형태입니다.

이런 분포에서는 **평균이 중앙값보다 큽니다.** 위쪽 극단값이 평균을 끌어올리기 때문입니다.

In [ ]:
print("평균  :", round(trips["duration"].mean(), 2), "분")
print("중앙값:", round(trips["duration"].median(), 2), "분")

**결과 읽는 법** — 평균 14.35분, 중앙값 10.90분입니다. **"보통 얼마나 걸리나요?"라는 질문에는 중앙값이 더 정직한
답**입니다. 이 성질은 뒤에서 결측치를 채울 때(평균 vs 중앙값)와 [스케일링](https://github.com/karzit/temp/blob/master/glossary.md#scaling) 방식을 고를 때 다시 등장합니다.

---

## 5. 두 변수 보기 — 수치형 + 수치형

### `scatterplot` / `jointplot`

`scatterplot`은 점만 찍고, `jointplot`은 **가운데 산점도 + 위·오른쪽에 각 변수의 히스토그램**을
함께 그려줍니다. 두 변수의 관계와 각각의 분포를 한 번에 보고 싶을 때 유용합니다.

In [ ]:
sns.jointplot(data=trips, x="distance", y="duration", height=6, alpha=0.3)
plt.show()

**결과 읽는 법** — 거리가 멀수록 시간이 오래 걸린다는 **양의 상관관계**가 뚜렷합니다. 다만 완전한 직선은 아닌데,
같은 거리라도 막히는 시간대냐 아니냐에 따라 시간이 달라지기 때문입니다.
바로 이 "설명되지 않는 부분"을 `hour`, `weekday` 같은 변수로 메우는 것이 모델링의 목표입니다.

`alpha=0.3`은 점을 반투명하게 만드는 옵션입니다. **점이 수천 개 겹칠 때는 필수**입니다.
안 주면 겹친 부분이 새까맣게 뭉쳐서 밀도를 알 수 없습니다.

> #### figure-level 함수와 axes-level 함수
>
> seaborn 함수는 두 종류입니다.
>
> - **axes-level** (`countplot`, `histplot`, `boxplot`, `scatterplot`, `heatmap`):
>   지정한 축 하나에 그립니다. **`ax=` 인자를 받아** `subplots` 안에 넣을 수 있습니다.
> - **figure-level** (`jointplot`, `pairplot`, `catplot`, `relplot`, `displot`):
>   **Figure 전체를 스스로 만듭니다.** 그래서 `ax=` 인자가 없고, `subplots` 안에 넣을 수 없습니다.
>   크기는 `height=`, `aspect=`로 조절합니다.
>
> `sns.jointplot(..., ax=axes[0])`은 `TypeError`가 납니다. `subplots` 안에 산점도를 넣고 싶다면
> axes-level인 `scatterplot`을 쓰면 됩니다.

In [ ]:
# jointplot 대신 scatterplot을 쓰면 subplots 안에 넣을 수 있습니다
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.scatterplot(data=trips, x="distance", y="duration", alpha=0.3, ax=axes[0])
axes[0].set_title("거리 - 시간")

sns.scatterplot(data=trips, x="distance", y="fare", alpha=0.3, ax=axes[1])
axes[1].set_title("거리 - 요금")

plt.tight_layout()
plt.show()

**결과 읽는 법** — 오른쪽 그래프에 **일정한 기울기의 직선**이 선명하게 보입니다. 뉴욕 택시 요금이 거리에 비례하는
정해진 요율로 계산되기 때문입니다. 직선에서 벗어난 점들은 대기 시간이 길었거나 통행료가 붙은 경우입니다.

## 6. 여러 변수 한 번에 — 상관계수 히트맵

**상관계수(correlation coefficient)** 는 두 수치형 변수가 얼마나 함께 움직이는지를 -1 ~ +1로 나타낸 값입니다.

- **+1에 가까움**: 하나가 커지면 다른 것도 커짐
- **0에 가까움**: 직선 관계가 없음
- **-1에 가까움**: 하나가 커지면 다른 것은 작아짐

`corr()`는 모든 수치형 컬럼 쌍에 대해 이 값을 계산합니다. **`numeric_only=True`를 반드시 넣어야**
문자열 컬럼 때문에 에러가 나지 않습니다.

In [ ]:
corr = trips.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
# heatmap: 표의 숫자를 색으로 / annot=True는 칸 안에 숫자도 함께 표시
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("택시 데이터 상관계수")
plt.show()

**결과 읽는 법**

`annot=True`는 각 칸에 숫자를 찍고, `center=0`은 0을 색상의 중앙(흰색)에 놓아 양수/음수를
색으로 구분하게 합니다.

예측 대상인 `duration`과의 상관만 따로 뽑아 정렬해보면, 어떤 변수가 유망한지 한눈에 보입니다.

In [ ]:
corr["duration"].drop("duration").sort_values(key=abs, ascending=False)

**결과 읽는 법**

`fare`(0.85)와 `distance`(0.82)가 압도적입니다. 당연한 결과지만 중요한 확인입니다 —
**상식과 데이터가 일치한다면 데이터를 제대로 불러온 것**입니다. 여기서 엉뚱한 결과가 나온다면
파생 변수 계산이나 컬럼 해석에 실수가 있다는 신호입니다.

다만 **요금이 거리보다도 상관이 높다**는 점은 기억해두세요. 03번에서 이것이 데이터 누출의
단서가 됩니다.

> #### 상관계수를 볼 때 주의할 점
>
> 1. **직선 관계만 잡아냅니다.** U자 모양처럼 뚜렷한 관계가 있어도 상관계수는 0에 가까울 수 있습니다.
> 2. **인과관계가 아닙니다.** `fare`와 `duration`이 함께 커지는 것은 둘 다 `distance`의 영향을 받기 때문이지,
>    요금이 시간을 만드는 것이 아닙니다.
> 3. **너무 높은 상관(0.9 이상)은 경고 신호입니다.** 같은 정보를 담은 중복 컬럼이거나, 심하면
>    정답에서 거꾸로 계산된 컬럼일 수 있습니다.

### 정답이 새어 들어간 컬럼 (데이터 누출)

`trips`에는 위험한 컬럼이 하나 있습니다. `speed`는 `distance / duration`으로 **우리가 직접
`duration`에서 계산해 만든 값**입니다. 즉 정답을 알아야만 만들 수 있는 컬럼입니다.

이런 컬럼을 피처로 넣으면 모델은 `distance / speed`로 정답을 그대로 복원해버립니다.
검증 점수는 완벽하게 나오지만, 실제로는 아무것도 배우지 못한 상태입니다.
이것을 **데이터 누출(data leakage)** 이라고 합니다.

**"이 값을 예측 시점에 실제로 알 수 있는가?"** 를 기준으로 판단하면 됩니다. 택시가 출발하는 순간에
평균 시속은 알 수 없습니다. 그러니 피처로 쓸 수 없습니다.

> 이 질문은 데이터가 바뀌어도 그대로 적용됩니다. **2부에서 타이타닉에도 똑같이 던져봅니다.**

In [ ]:
# 실제로 얼마나 완벽하게 복원되는지 확인
recovered = trips["distance"] / (trips["speed"] / 60)
print("복원값과 실제 duration의 상관계수:", round(recovered.corr(trips["duration"]), 6))

**결과 읽는 법** — 상관계수 1.0. `speed`를 넣는 순간 모델은 이 나눗셈만 배우면 끝입니다.

## 7. 그래프를 나란히 놓고 비교하기 — `plt.subplots`

이미 위에서 여러 번 썼지만, 문법을 정리하고 넘어갑니다.

```python
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 4))
```

- **`nrows` × `ncols`** 만큼 격자를 만듭니다. **가로로 나란히 = `nrows=1, ncols=2`**, 세로로 쌓기 = `nrows=2, ncols=1`
- `axes`는 축들의 배열입니다. `axes[0]`, `axes[1]`로 접근하고, 2행 2열이면 `axes[0][1]`처럼 2차원입니다
- 각 그래프를 어느 칸에 그릴지는 **`ax=axes[0]`** 으로 지정합니다. 빼먹으면 전부 한 칸에 겹쳐 그려집니다
- 축이 여러 개일 때 제목은 `plt.title()`이 아니라 **`axes[i].set_title()`** 입니다
  (`plt.title()`은 마지막에 활성화된 축 하나에만 붙습니다)
- `plt.tight_layout()`은 라벨이 서로 겹치지 않게 간격을 자동 조정합니다

라벨이 길어 겹칠 때는 기울입니다.

```python
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")
```

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(13, 9))

sns.countplot(data=trips, x="hour", ax=axes[0][0])
axes[0][0].set_title("시간대별 운행 건수")

sns.boxplot(data=trips, x="weekday", y="duration", ax=axes[0][1])
axes[0][1].set_title("요일별 이동 시간 (0=월)")

sns.histplot(data=trips, x="distance", bins=50, ax=axes[1][0])
axes[1][0].set_title("이동 거리 분포")

sns.countplot(data=trips, y="pickup_zone",
              order=trips["pickup_zone"].value_counts().head(10).index, ax=axes[1][1])
axes[1][1].set_title("승차 지역 Top 10")

plt.tight_layout()
plt.show()

**결과 읽는 법** — 오른쪽 아래 그래프에서 `order=`를 썼습니다. **`value_counts()`로 순서를 정해 넘기면 상위 N개만
골라 정렬된 상태로 그릴 수 있습니다.** 범주가 194개인 `pickup_zone`을 그냥 그리면 읽을 수 없으므로
꼭 필요한 기법입니다. 범주 이름이 길어서 `y=`로 가로 막대를 쓴 것도 같은 이유입니다.

시간대 그래프를 보면 새벽 5시경이 가장 한산하고 저녁 6~7시가 붐빕니다.
**`hour`가 이동 시간 예측에 기여할 만하다**는 근거입니다.

---

## 8. 1부 정리 — 택시 전처리 계획

여기까지 관찰한 것을 표로 정리하면, 그대로 다음 노트북의 작업 목록이 됩니다.
**EDA의 진짜 결과물은 그래프가 아니라 이 표입니다.**

### `trips` (회귀)

| 관찰 | 판단 | 처리 |
|---|---|---|
| `speed` max = 4834, `duration` min = 0 | 기록 오류 | **이상치 행 제거** |
| `payment` 등 5개 컬럼에 결측치 (0.4~0.7%) | 비율이 낮음 | **해당 행 제거** |
| `color`, `payment`, `*_borough`가 `object` | 모델에 못 넣음 | **[원-핫 인코딩](https://github.com/karzit/temp/blob/master/glossary.md#one-hot-encoding)** |
| `pickup_zone`이 고유값 194개 | 인코딩하면 컬럼 폭증 | **컬럼 삭제** (`*_borough`로 대체) |
| `speed`는 `duration`에서 계산됨 | 데이터 누출 | **컬럼 삭제** |
| `total` = `fare` + `tip` + `tolls` | 중복 정보 | **컬럼 삭제** |
| `pickup`/`dropoff`가 `datetime` | 그대로는 못 씀 | **컬럼 삭제** (`weekday`/`hour`로 대체) |

**1부에서 한 일**: 첫인상 파악 → 그래프로 분포와 관계 확인 → 상관계수로 유망한 변수 추리기 →
데이터 누출 걸러내기 → 계획표 작성. **이것이 EDA의 전부**이고, 어떤 데이터를 받아도 순서는 같습니다.

이제 데이터를 바꿔서, 택시로는 볼 수 없었던 것을 봅니다.

---

# 2부. 타이타닉 — 한 단계 더

**1부에서 익힌 흐름은 그대로입니다.** 택시 데이터로는 연습할 수 없었던 세 가지를 여기서 채웁니다.

| 택시에서는 | 타이타닉에서는 |
|---|---|
| 결측치가 1%대 — 그냥 지우면 끝 | **`deck`이 77% 결측** — 채울지 버릴지 판단이 필요 |
| 범주가 자치구·결제 수단뿐이라 대비가 약함 | **생존 여부·객실 등급** — 범주별 차이가 극적으로 드러남 |
| 누출 컬럼이 우리가 직접 만든 `speed` | **원본에 이미 들어 있는 `alive`** — 남이 만든 데이터에서 찾아내기 |

예측 대상은 `survived`(0=사망, 1=생존)이고 **분류 문제**입니다. 03번에서 이 데이터로 분류 모델을 만듭니다.

## 9. 타이타닉 불러오기

In [ ]:
# [분류용] 타이타닉 탑승자 — 생존 여부(0/1)를 맞히는 문제로 씁니다.
titanic = sns.load_dataset("titanic")

print("titanic:", titanic.shape)
titanic.head()

### `titanic` 컬럼 설명

| 컬럼 | 설명 |
|---|---|
| `survived` | 생존 여부 (0=사망, 1=생존) — **예측 대상** |
| `pclass` / `class` | 객실 등급 (1/2/3 숫자 · First/Second/Third 문자) — 같은 정보 |
| `sex` / `who` / `adult_male` | 성별 / man·woman·child / 성인 남성 여부 |
| `age` | 나이 |
| `sibsp` / `parch` / `alone` | 동반 형제·배우자 수 / 동반 부모·자녀 수 / 혼자 탑승 여부 |
| `fare` | 요금 |
| `embarked` / `embark_town` | 탑승 항구 (S/C/Q 약자 · 지명) — 같은 정보 |
| `deck` | 갑판 |
| `alive` | 생존 여부 (yes/no) — `survived`와 **같은 정보** |

---

## 10. 결측치가 심각할 때 — 채울까, 버릴까

`info()`를 읽는 법은 1부에서 익혔습니다. 여기서 주제는 **결측치가 많을 때 무엇을 근거로
판단하는가**입니다. 같은 `info()`를 봐도 택시와는 나오는 결론이 다릅니다.

In [ ]:
titanic.info()

**결과 읽는 법** — 개수만 보면 감이 오지 않습니다. **전체 대비 비율(%)을 함께 봐야** 판단할 수 있습니다.
아래 표는 결측치가 있는 컬럼만 골라 개수와 비율을 나란히 보여줍니다.

In [ ]:
# 결측치 개수와 비율을 함께 보기 — 비율이 판단에 더 유용합니다
missing = pd.DataFrame({
    "결측치 개수": titanic.isnull().sum(),
    "비율(%)": (titanic.isnull().mean() * 100).round(1),
})
missing[missing["결측치 개수"] > 0]

**결과 읽는 법**

`deck`은 **77.2%가 비어 있습니다.** 이 정도면 채워 넣어도 대부분이 추측값이라 의미가 없으므로,
보통 컬럼 자체를 버립니다. 반면 `age`는 19.9%라 채워 쓸지 지울지 고민해볼 만합니다.
이 판단 기준은 다음 노트북에서 자세히 다룹니다.

---

## 11. 범주형 + 수치형 — `boxplot`

1부에서는 수치형 두 개의 관계(거리-시간)를 봤습니다. 이번엔 **범주에 따라 수치가 어떻게
달라지는가**입니다. 3절 그래프 표의 "범주형 + 수치형" 줄에 해당합니다.

박스플롯은 정보 밀도가 아주 높은 그래프입니다. 구성 요소를 정확히 알아두면 다음 노트북의
이상치 처리가 훨씬 쉬워집니다.

```
     ┬     ← 위쪽 수염 끝 = upper fence (Q3 + 1.5 × IQR) 안에 있는 실제 최댓값
     │
   ┌─┴─┐   ← Q3 (75% 지점)
   │───│   ← 중앙값 (50% 지점)
   └─┬─┘   ← Q1 (25% 지점)
     │
     ┴     ← 아래쪽 수염 끝 = lower fence (Q1 - 1.5 × IQR) 안에 있는 실제 최솟값

     ·  ·  ← 수염 밖의 점 = 이상치 후보
```

- **[IQR](https://github.com/karzit/temp/blob/master/glossary.md#iqr)(Interquartile Range, 사분위 범위) = Q3 - Q1** — 가운데 50%가 퍼진 폭
- 박스가 짧으면 값이 몰려 있고, 길면 넓게 퍼져 있습니다
- **수염 밖의 점**들이 통계적으로 이상치로 간주되는 값입니다

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(data=titanic, x="survived", y="fare", ax=axes[0])
axes[0].set_title("생존 여부별 요금")

sns.boxplot(data=titanic, x="pclass", y="age", ax=axes[1])
axes[1].set_title("객실 등급별 나이")

plt.tight_layout()
plt.show()

**결과 읽는 법**

- **왼쪽**: 생존자(1) 쪽 요금 중앙값이 확실히 높습니다. 요금이 비쌀수록 좋은 객실이었고, 좋은 객실은
  갑판에 가까웠습니다. 양쪽 모두 위로 이상치가 길게 뻗어 있어 `fare`에 이상치 처리가 필요해 보입니다.
- **오른쪽**: 1등급 승객이 3등급보다 나이가 많습니다. 부를 축적할 시간이 필요했다는 뜻이겠죠.

박스플롯이 보여주는 "차이가 있다"는 곧 **"이 변수는 예측에 도움이 된다"** 는 신호입니다.

---

## 12. 그룹별로 나눠 보기 — `hue`

거의 모든 seaborn 함수에 `hue=`를 추가하면 범주별로 색을 나눠 겹쳐 그립니다.
**"A별 B의 분포"** 라는 표현이 나오면 `hue=A, x=B`입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data=titanic, x="age", hue="survived", bins=30, ax=axes[0])
axes[0].set_title("생존 여부별 나이 분포")

sns.countplot(data=titanic, x="sex", hue="survived", ax=axes[1])
axes[1].set_title("성별 생존 여부")

plt.tight_layout()
plt.show()

**결과 읽는 법** — 오른쪽 그래프의 차이가 극적입니다. 숫자로 확인해봅시다. `groupby`로 묶고 `survived`의 평균을 내면,
0과 1로만 이루어진 값의 평균이므로 **그대로 생존율**이 됩니다.

In [ ]:
print("성별 생존율")
print((titanic.groupby("sex")["survived"].mean() * 100).round(1))
print()
print("객실 등급별 생존율")
print((titanic.groupby("pclass")["survived"].mean() * 100).round(1))

**결과 읽는 법** — 여성 74.2% vs 남성 18.9%, 1등급 63.0% vs 3등급 24.2%. **`sex`와 `pclass`는 생존을 예측하는 데
매우 강력한 변수**라는 뜻이고, 실제로 03번 노트북의 변수중요도에서도 상위권에 나타납니다.

---

## 13. 여기서도 데이터 누출 — `alive`

1부에서 `speed`를 걸러낸 질문을 그대로 던져봅니다. **"이 값을 예측 시점에 알 수 있는가?"**

`titanic`의 `alive`는 `survived`를 `yes`/`no`로 바꿔놓은 것뿐입니다. 생존 여부를 맞히는 모델에
생존 여부를 그대로 알려주는 셈이라, 정확도는 100%가 나오고 **모델은 아무것도 배우지 못합니다.**

`speed`와 다른 점은 **우리가 만든 컬럼이 아니라는 것**입니다. 원본에 처음부터 들어 있었습니다.
남이 만든 데이터를 받았을 때 **컬럼 설명을 한 줄씩 읽어봐야 하는 이유**입니다.

In [ ]:
# alive가 정말 survived와 같은 정보인지 확인
print(pd.crosstab(titanic["survived"], titanic["alive"]))   # crosstab: 두 컬럼의 값 조합별 개수를 교차표로 만든다
print()
print("완전히 일치하는가:",
      ((titanic["alive"] == "yes").astype(int) == titanic["survived"]).all())

**결과 읽는 법** — 대각선 밖이 전부 0입니다. 두 컬럼은 표기만 다른 같은 값입니다.

> 비슷한 이유로 **같은 정보를 두 번 담은 컬럼**도 정리 대상입니다. `class`(First/Second/Third)는
> `pclass`(1/2/3)와 같고, `embark_town`은 `embarked`의 지명 버전입니다. 누출은 아니지만
> 넣어봐야 얻는 것이 없습니다.

---

## 14. 2부 정리 — 타이타닉 전처리 계획

1부와 똑같이, 관찰한 것을 계획표로 옮깁니다.

### `titanic` (분류)

| 관찰 | 판단 | 처리 |
|---|---|---|
| `fare`에 위쪽 이상치 다수 | 실제 값이지만 극단적 | **IQR 기준으로 제거** |
| `deck` 결측 77.2% | 채워도 의미 없음 | **컬럼 삭제** |
| `age` 결측 19.9% | 중요한 변수 | **행 제거 또는 중앙값 대체** |
| `alive`가 `survived`와 동일 | 데이터 누출 | **컬럼 삭제** |
| `class`/`pclass`, `embark_town`/`embarked` 중복 | 같은 정보 | **한쪽만 남기기** |
| `sex`, `embarked`, `who`가 `object` | 모델에 못 넣음 | **원-핫 인코딩** |

---

## 정리

**1부 — 택시 (EDA의 기본 흐름)**

- **`info()`는 결측치와 자료형을, `describe()`는 이상치를 알려줍니다.** 이 둘이 전처리 계획의 출발점입니다
- 그래프는 **변수 개수 × 변수 유형**으로 고릅니다: 범주 하나→`countplot`, 수치 하나→`histplot`,
  수치+수치→`scatterplot`/`jointplot`, 여러 수치→`heatmap`
- **figure-level 함수(`jointplot`, `pairplot`)는 `ax=`를 받지 않습니다**
- **오른쪽으로 치우친 분포에서는 평균 > 중앙값** — 결측치 대체와 스케일링 선택에 영향을 줍니다
- **"예측 시점에 알 수 있는 값인가"** 로 데이터 누출을 걸러냅니다 (`speed`)

**2부 — 타이타닉 (한 단계 더)**

- **결측 비율이 판단 기준입니다.** 77%면 컬럼째 버리고, 20%면 채울지 지울지 고민합니다
- **범주 + 수치는 `boxplot`**, 여기에 `hue=`를 더하면 그룹별로 나눠 볼 수 있습니다.
  차이가 크게 벌어진다면 **그 변수는 예측에 도움이 된다**는 신호입니다
- 같은 누출 질문이 다른 데이터에도 그대로 적용됩니다 (`alive`).
  다만 이번엔 **원본에 처음부터 들어 있던 컬럼**이라 컬럼 설명을 읽지 않으면 놓칩니다

## 스스로 확인해보기

- [ ] `info()`와 `describe()`가 각각 무엇을 알려주는지 구분할 수 있다
- [ ] 변수의 개수와 유형(범주형/수치형)으로 그래프를 고를 수 있다
- [ ] 오른쪽으로 치우친 분포에서 평균과 중앙값 중 무엇을 써야 할지 판단할 수 있다
- [ ] **"예측 시점에 알 수 있는 값인가"** 로 데이터 누출을 걸러낼 수 있다
- [ ] 결측 비율을 보고 채울지 버릴지 판단할 수 있다
- [ ] `jointplot`·`pairplot`에 `ax=`를 넘기면 왜 안 되는지 안다
- [ ] EDA의 결과물이 "예쁜 그래프"가 아니라 **전처리 계획표**라는 것을 이해했다

## 연습 문제

풀어본 뒤 [01_eda_visualization_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/01_eda_visualization/01_eda_visualization_solutions.ipynb)에서 확인하세요.
**문제 1~4는 1부(택시), 문제 5~6은 2부(타이타닉)** 범위입니다.

### 1부 — 택시

**문제 1.** `trips`에서 하차 자치구(`dropoff_borough`)의 분포를 countplot으로 그리고,
`value_counts()`로 각 자치구의 비율(%)을 구하세요. 승차 자치구 분포와 비교했을 때
눈에 띄는 차이가 있나요?

**문제 2.** `trips`에서 결제 수단(`payment`)에 따라 팁(`tip`) 분포가 어떻게 다른지 보여주는
그래프를 그리고, 두 그룹의 **평균과 중앙값**을 각각 구하세요. 왜 이런 차이가 나타날까요?

**문제 3.** 아래 코드는 세 군데가 잘못되어 원하는 그림이 나오지 않습니다. 무엇이 문제인지 찾아
고치세요. (의도: 승차 자치구별 운행 수와 거리-시간 산점도를 가로로 나란히 그리고, 왼쪽에 제목 달기)

```python
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 4))
sns.countplot(data=trips, x="pickup_borough", ax=axes[1])
sns.jointplot(data=trips, x="distance", y="duration", ax=axes[2])
axes[0].title("승차 자치구")
plt.show()
```

**문제 4.** `trips`에서 `passengers`가 0인 행이 96건 있습니다. `describe()`와 그래프로
이 행들이 다른 행과 어떻게 다른지 살펴보고, 제거해야 할지 판단해보세요.
(정답이 하나로 정해진 문제는 아닙니다. 근거를 세워보는 것이 목적입니다.)

### 2부 — 타이타닉

**문제 5.** `titanic`에서 아래 두 그래프를 **가로로 나란히** 그리세요.
- 왼쪽: 탑승 항구(`embark_town`)별 개수를 보여주는 countplot, 제목 `"탑승 항구"`, X축 라벨 45도 기울임
- 오른쪽: 객실 등급(`pclass`)별 요금(`fare`) 분포를 보여주는 boxplot, 제목 `"등급별 요금"`

**문제 6.** `titanic`의 수치형 컬럼들로 상관계수 히트맵을 그리고, `survived`와 상관계수의
절댓값이 가장 큰 컬럼을 찾으세요. 그 결과가 2부에서 본 성별·등급별 생존율과 어떻게 연결되나요?

---

다음 노트북: [02_preprocessing.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/02_preprocessing/02_preprocessing.ipynb) — 여기서 만든
계획표대로 이상치·결측치·인코딩·스케일링을 처리합니다.